In [2]:
# CELL 1: Ingestion
import numpy as np
import pandas as pd

# Load dataset using latin-1 encoding to prevent byte decode errors
file_path = "DataCoSupplyChainDataset.csv"
df = pd.read_csv(file_path, encoding="latin-1")

print(f"Dataset successfully loaded.")
print(f"Initial Dimensions: {df.shape[0]} rows and {df.shape[1]} columns")
df.head(6)

Dataset successfully loaded.
Initial Dimensions: 180519 rows and 53 columns


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class
5,TRANSFER,6,4,18.580000,294.980011,Shipping canceled,0,73,Sporting Goods,Tonawanda,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/19/2018 11:03,Standard Class


In [3]:
# CELL 2: Missing Value Audit
# Calculate missing count and percentage
missing_count = df.isnull().sum()
missing_percent = (missing_count / len(df)) * 100

# Collate into an audit DataFrame
audit_df = pd.DataFrame(
    {"Missing_Count": missing_count, "Missing_Percentage": missing_percent}
)

# Display only columns that actually have missing data
missing_summary = audit_df[audit_df["Missing_Count"] > 0].sort_values(
    by="Missing_Count", ascending=False
)
print("--- COLUMNS WITH MISSING DATA ---")
print(missing_summary)

--- COLUMNS WITH MISSING DATA ---
                     Missing_Count  Missing_Percentage
Product Description         180519          100.000000
Order Zipcode               155679           86.239676
Customer Lname                   8            0.004432
Customer Zipcode                 3            0.001662


In [4]:
# CELL 3: Structural Pruning & Specific Missing Value Resolution
# 1. Drop 100% empty, high-missing unrecoverable spatial columns, and PII
cols_to_drop = [
    "Product Description",  # 100% missing
    "Order Zipcode",  # >85% missing, redundant with Order City/Country
    "Product Image",  # Non-analytical URL string
    "Customer Email",  # PII
    "Customer Password",  # PII & security hazard
]

df.drop(columns=cols_to_drop, inplace=True)

# 2. Impute minor categorical/geographic fields with fallback placeholders
# Customer Zipcode has only ~3 missing rows; impute with fallback '00000'
df["Customer Zipcode"] = df["Customer Zipcode"].fillna(0).astype(int)
df["Customer Lname"] = df["Customer Lname"].fillna("Unknown")

print("Target columns pruned successfully.")
print(f"Updated dataset dimensions: {df.shape}")

Target columns pruned successfully.
Updated dataset dimensions: (180519, 48)


In [5]:
# CELL 4: Temporal Standardization
# Convert timestamps to native pandas datetime
df["order date (DateOrders)"] = pd.to_datetime(df["order date (DateOrders)"])
df["shipping date (DateOrders)"] = pd.to_datetime(
    df["shipping date (DateOrders)"]
)

# Chronological validation check
invalid_chronology = (
    df["shipping date (DateOrders)"] < df["order date (DateOrders)"]
).sum()
print(f"Invalid dispatch records found: {invalid_chronology}")

# Retain only chronologically valid records
df = df[
    df["shipping date (DateOrders)"] >= df["order date (DateOrders)"]
].copy()

Invalid dispatch records found: 0


In [6]:
# CELL 5: Outlier Capping via IQR
outlier_features = [
    "Days for shipping (real)",
    "Sales per customer",
    "Order Item Product Price",
]

for col in outlier_features:
  q1 = df[col].quantile(0.25)
  q3 = df[col].quantile(0.75)
  iqr = q3 - q1
  lower_bound = q1 - (1.5 * iqr)
  upper_bound = q3 + (1.5 * iqr)

  # Check how many data points exist outside bounds
  outliers_count = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
  print(f"{col}: {outliers_count} outliers detected outside bounds. Capping...")

  # Winsorize using pandas clip
  df[col] = df[col].clip(
      lower=max(0, lower_bound), upper=upper_bound
  )  # Keep lower bound >= 0 for prices/days

Days for shipping (real): 0 outliers detected outside bounds. Capping...
Sales per customer: 1943 outliers detected outside bounds. Capping...
Order Item Product Price: 2048 outliers detected outside bounds. Capping...


In [7]:
# CELL 6: Feature Engineering
# Calculate internal warehouse processing / dispatch delay (in days)
df["Dispatch_Lead_Days"] = (
    df["shipping date (DateOrders)"] - df["order date (DateOrders)"]
).dt.total_seconds() / 86400.0

# Extract operational dispatch day-of-week (0=Monday, 6=Sunday)
df["Dispatch_Day_Of_Week"] = df["shipping date (DateOrders)"].dt.dayofweek
df["Is_Weekend_Dispatch"] = df["Dispatch_Day_Of_Week"].isin([5, 6]).astype(int)

# Delivery delay variance against scheduled SLA target
df["Delivery_SLA_Variance"] = (
    df["Days for shipping (real)"] - df["Days for shipment (scheduled)"]
)

print("Engineered domain variables:")
print(
    df[
        [
            "Dispatch_Lead_Days",
            "Is_Weekend_Dispatch",
            "Delivery_SLA_Variance",
            "Late_delivery_risk",
        ]
    ].head()
)

Engineered domain variables:
   Dispatch_Lead_Days  Is_Weekend_Dispatch  Delivery_SLA_Variance  \
0                 3.0                    1                     -1   
1                 5.0                    0                      1   
2                 4.0                    0                      0   
3                 3.0                    0                     -1   
4                 2.0                    0                     -2   

   Late_delivery_risk  
0                   0  
1                   1  
2                   0  
3                   0  
4                   0  


In [8]:
# CELL 7: Scaling & Encoding using Pandas
# 1. Z-Score Standardization on numeric features
numeric_to_scale = [
    "Sales per customer",
    "Order Item Product Price",
    "Dispatch_Lead_Days",
    "Days for shipping (real)",
]

for col in numeric_to_scale:
  mean_val = df[col].mean()
  std_val = df[col].std()
  df[f"{col}_Scaled"] = (df[col] - mean_val) / std_val

# 2. One-Hot Encode Shipping Mode
shipping_dummies = pd.get_dummies(
    df["Shipping Mode"], prefix="Mode", drop_first=True, dtype=int
)

# Merge back into the final dataframe
df_clean = pd.concat([df, shipping_dummies], axis=1)

print("Scaling and One-Hot Encoding completed.")

Scaling and One-Hot Encoding completed.


In [9]:
# CELL 8: Final Verification & Export
print("--- PRE-PROCESSING VERIFICATION ---")
print("Remaining null values in analytical columns:")
key_cols = [
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Sales per customer",
    "Dispatch_Lead_Days",
    "Late_delivery_risk",
]
print(df_clean[key_cols].isnull().sum())

print("\n--- STATISTICAL SUMMARY OF CLEANED ATTRIBUTES ---")
print(df_clean[key_cols].describe())

# Save clean CSV for Week 3 visualizations
df_clean.to_csv("Cleaned_DataCo_Logistics.csv", index=False)
print("\nExported clean file to 'Cleaned_DataCo_Logistics.csv'.")

--- PRE-PROCESSING VERIFICATION ---
Remaining null values in analytical columns:
Days for shipping (real)         0
Days for shipment (scheduled)    0
Sales per customer               0
Dispatch_Lead_Days               0
Late_delivery_risk               0
dtype: int64

--- STATISTICAL SUMMARY OF CLEANED ATTRIBUTES ---
       Days for shipping (real)  Days for shipment (scheduled)  \
count             180519.000000                  180519.000000   
mean                   3.497654                       2.931847   
std                    1.623722                       1.374449   
min                    0.000000                       0.000000   
25%                    2.000000                       2.000000   
50%                    3.000000                       4.000000   
75%                    5.000000                       4.000000   
max                    6.000000                       4.000000   

       Sales per customer  Dispatch_Lead_Days  Late_delivery_risk  
count       18051